# Grid Measure — **통신 개선판 v2**

[grid_measure_config.ipynb](grid_measure_config.ipynb) 복사 + **통신(연결) 부분만** 개선:
- 연결 시 **GPIB Interface Clear(IFC)** → NCIC(제어권 상실) 예방
- **핸드셰이크 타임아웃 8초** → 응답 없으면 10분 대신 몇 초 안에 실패
- B1500 `*IDN?` 확인 후 초기화, 실패 시 원인 안내
- `drain_b1500_errors` 통신오류 시 즉시 종료(무한대기 방지)

> ⚠️ 근본 해결 = **NI MAX → GPIB0 → 'System Controller' 체크**. 측정/이동/좌표는 원본과 동일.

## ⚠️ 코드 실행 전 — 사람이 손으로 끝내놔야 하는 준비 (Nucleus UI)
매뉴얼 워크플로우 중 이 단계들은 **코드가 대체하지 않으므로** 먼저 수동으로 완료해야 함:
1. 척 로드 / 진공 ON / 소자 세팅
2. **Alignment** (2-point align) — 안 하면 좌표가 통째로 어긋남
3. **Tipping / Set Contact** — 팁 contact 높이를 잡아둠
4. **첫 소자에 팁을 직접 contact** 시킨 상태로 둠 → 아래 5번 셀에서 그 위치를 원점·contact높이로 등록

> 측정 중 light off 등은 Nucleus 쪽 물리 작업이라 코드 밖.

In [1]:
import os
import time
import pandas as pd
import pyvisa
from pymeasure.instruments.agilent import AgilentB1500

# --- pandas 3.0 호환 shim -----------------------------------------------------
# pandas 2.1 에서 DataFrame.applymap -> DataFrame.map 으로 이름 바뀌고
# pandas 3.0 에서 applymap 이 제거됨. 그런데 pymeasure 0.16 의 B1500 read_data 가
# 아직 applymap 을 써서 측정 데이터 읽을 때 에러남. map 이 applymap 과 동작 동일하므로
# 옛 이름을 다시 연결해 호환을 살린다. (라이브러리가 pandas 3 지원하면 삭제 가능)
if not hasattr(pd.DataFrame, "applymap"):
    pd.DataFrame.applymap = pd.DataFrame.map

## 0. 설정 (여기만 바꾸면 됨)

In [2]:
# --- 장비 주소 ---
S300_GPIB  = "GPIB0::28::INSTR"
B1500_GPIB = "GPIB0::17::INSTR"

# --- 좌표 CSV (형식: Subsite Name, X Position, Y Position, Note), 단위 = micron ---
# 테스트용(3점, 최대 100um 이동). 실제 측정 시 아래 old 경로로 되돌릴 것.
# COORD_CSV = "../../utils/test_coordinates.csv"   # 테스트용(3점)
COORD_CSV = "../../utils/grid_2x4.csv"              # 실제 측정용 (2x4 = 8점)
# COORD_CSV = "../../utils/old/grid_coordinates.csv"   # 실제 측정용 (160점)

# --- 드라이브할 전압 범위 (drive="sweep" 인 SMU 에 적용) ---
V_START      = 0.0     # 시작 전압 [V]
V_STOP       = 1.0     # 끝 전압 [V]
V_POINTS     = 11      # sweep 포인트 수
I_COMPLIANCE = 1e-3    # 전류 컴플라이언스 [A]

# --- 팁(프로브) ↔ SMU ↔ 역할 매핑 -------------------------------------------
# 팁 4개가 각각 어느 단자(Gate/Drain/Source/Bulk)에 닿는지는 셋업마다 다르므로
# 여기서 직접 지정한다.  키 N = B1500 채널 번호 smuN  (UNT? 로 장착 확인).
#   role    : 표기용 이름 (자유)
#   drive   : "sweep" → V_START~V_STOP sweep (보통 Gate)
#             "const" → const_v 로 DC 고정 (보통 Drain/Source/Bulk)
#             "off"   → 사용 안 함(disable)
#   const_v : drive="const" 일 때 인가 전압 [V]
SMU_CONFIG = {
    1: {"role": "GATE",   "drive": "sweep", "const_v": None},
    2: {"role": "DRAIN",  "drive": "const", "const_v": 1.0},
    3: {"role": "SOURCE", "drive": "const", "const_v": 0.0},
    4: {"role": "BULK",   "drive": "off",   "const_v": 0.0},
}

# --- S300 척(chuck) Device ID ---
# 매뉴얼 p.37: 2 = Elite/Summit/S300/Alessi 의 척 제어 device ID (축 번호 아님)
CHUCK_ID = 2

# --- 결과 저장 폴더 ---
OUT_DIR = "results"

## 0-C. Transfer / Output 측정 파라미터

- **Transfer curve (Id-Vg):** 게이트를 sweep, 드레인 고정 → 문턱전압·on/off 특성
- **Output curve (Id-Vd):** 드레인을 sweep + 게이트를 여러 값(family) → 출력 특성
채널은 `SMU_CONFIG` 의 role(GATE/DRAIN/SOURCE)에서 자동으로 잡습니다.

In [ ]:
# 채널 자동 추출 (SMU_CONFIG 의 role 기준)
GATE_CH   = next(ch for ch, c in SMU_CONFIG.items() if c["role"] == "GATE")
DRAIN_CH  = next(ch for ch, c in SMU_CONFIG.items() if c["role"] == "DRAIN")
SOURCE_CH = next(ch for ch, c in SMU_CONFIG.items() if c["role"] == "SOURCE")

# --- Transfer curve (Id-Vg): 실제 조건 (ID_VG_CEC 재현) ---
VG_START, VG_STOP, VG_POINTS = 0.0, 10.0, 81    # 게이트 0~10V, 125mV step → 81점 (왕복)
VD_LIST = [0.1, 2.0, 3.9]                       # 드레인 스텝: 0.1 / 2.0 / 3.9 V (3개)
TR_COMPLIANCE = 10e-3                           # 컴플라이언스 10 mA
TR_MEAS_RANGE = "1 nA"                          # export: LIMITED 1nA (1nA 하한 오토레인지)

# --- Output curve (Id-Vd): 실제 측정 조건 (ID_Vd_CEC 셋업 재현) ---
VD_START, VD_STOP, VD_POINTS = 0.0, 10.0, 81    # 드레인 0~10V, 125mV step → 81점 (왕복)
VG_LIST = [10.0 - 0.5*k for k in range(15)]     # 게이트 10V→3V, -0.5V, 15 스텝
OUT_COMPLIANCE = 10e-3                           # 컴플라이언스 10 mA
OUT_MEAS_RANGE = "1 nA"                          # export: LIMITED 1nA (1nA 하한 오토레인지)

print(f"채널: GATE=SMU{GATE_CH}, DRAIN=SMU{DRAIN_CH}, SOURCE=SMU{SOURCE_CH}")

## 0-B. B1500 Measurement Setup 파라미터 (스크린샷 UI ↔ 코드)

위 `0.` 설정은 **원본 그대로**(원본 `measure_iv`/`measure_sampling` 용). 
아래는 **추가된 설정판** — EasyEXPERT 의 *Measurement Setup* 탭 손잡이를 그대로 노출한다.
이 값들은 아래 `## 2-B` 의 `measure_iv_full()` 이 사용한다. (원본 함수엔 영향 없음)

In [3]:
# ============================================================================
# B1500 "Measurement Setup" 화면을 코드로 재현하는 설정값들
#   VAR1  : 주 sweep 소스   (스크린샷 1 - 파란박스 1)
#   VAR2  : 부 sweep 소스   (스크린샷 2 - 각 VAR2 스텝마다 VAR1 전체 sweep 반복)
#   Timing: Hold/Delay/Sweep(auto abort) (스크린샷 1 - 파란박스 2)
#   ADC   : 적분/분해능
#   Range : 채널별 측정 레인지 모드 (스크린샷 3 - Set Ranging Mode)
#   Const : DC 고정 소스는 위 SMU_CONFIG 의 const_v 를 그대로 사용 (스크린샷 1 - 파란박스 3)
# ============================================================================

# --- VAR1 (주 sweep) : 스크린샷 1 "Id-Vg Vth CT" 예시값 그대로 --------------
VAR1_CONFIG = {
    "unit":       1,          # sweep 할 SMU 채널 = SMU_CONFIG 의 키 (drive="sweep" 여야 함)
    "name":       "Vg",       # 표기용 (UI Name)
    "direction":  "single",   # "single"=편도 | "double"=왕복(hysteresis)   (UI Direction)
    "spacing":    "linear",   # "linear" | "log"                             (UI Linear/Log)
    "start":      -0.5,       # Start [V]   (UI: -500 mV)
    "stop":       3.0,        # Stop  [V]   (UI: 3 V)
    "points":     101,        # No of Step (포인트 수)
    "step":       None,       # Step [V]. 지정하면 points 대신 이걸로 계산 (UI Step 35 mV)
    "compliance": 100e-6,     # Compliance [A]   (UI: 100 uA)
    "power_comp": None,       # Pwr Comp [W] (None=OFF)
}

# --- VAR2 (부 sweep) : 스크린샷 2. 필요 없으면 None -------------------------
# 지정하면 VAR2 채널을 start~stop 로 stepping 하며 각 스텝마다 VAR1 sweep 전체를 반복한다.
# (Id-Vd 계열 특성처럼 2차원 sweep). VAR2 채널은 SMU_CONFIG 에서 drive="const" 여야 함.
VAR2_CONFIG = None
# 예) 드레인(SMU2)을 0.1V -> 1.0V 4스텝으로 올리며 각 스텝마다 Vg sweep:
# VAR2_CONFIG = {
#     "unit": 2, "name": "Vd",
#     "start": 0.1, "stop": 1.0, "points": 4, "step": None,
#     "compliance": 100e-3,
# }

# --- Timing : 스크린샷 1 - 파란박스 2 --------------------------------------
TIMING_CONFIG = {
    "hold":       0.0,        # Hold [s]  (UI: 0 s)
    "delay":      0.0,        # Delay [s] (UI: 0 s)
    "step_delay": 0.1,        # 스텝별 측정 지연 [s] (정착시간). UI엔 안 보이지만 WT 의 일부
    "auto_abort": False,      # Sweep 자동중단(compliance 등). UI "* Sweep CONTINUE AT ANY" = False
    "post":       "STOP",     # 측정 후 출력 유지 위치: "START"(시작값) | "STOP"(끝값)
}

# --- ADC (적분/분해능) -----------------------------------------------------
ADC_CONFIG = {
    "adc_type": "HRADC",      # "HRADC"(고분해능) | "HSADC"(고속)
    "mode":     "AUTO",       # "AUTO" | "MANUAL" | "PLC"
    "N":        6,            # 적분 계수 (HRADC AUTO 기본 6)
}

# --- Ranging Mode : 스크린샷 3 (Set Ranging Mode) --------------------------
# 채널별 "측정 전류 레인지".
#   mode : "auto"    = Auto Ranging (range 무시)
#          "limited" = 하한 지정 자동 (지정 레인지 밑으로는 안 내려감) = UI LIMITED
#          "fixed"   = 고정 레인지                                   = UI FIXED
#   range: "1 nA","10 nA","100 nA","1 uA","10 uA",... (UI Range 열)
RANGE_CONFIG = {
    1: {"mode": "limited", "range": "1 nA"},
    2: {"mode": "limited", "range": "1 nA"},
    3: {"mode": "limited", "range": "1 nA"},
    4: {"mode": "limited", "range": "1 nA"},
}


## 1. S300 프로버 제어 (① 코드)
`S300_test_v2.ipynb` 의 `CascadeS300` 에 매뉴얼 기반 편의 함수 추가.

**기준점/안전 동작 (매뉴얼 근거)**
- `set_reference()` : 사람이 첫 소자에 contact 시킨 현재 위치를 등록 — `:set:pres 2 0 0`(현재 XY=원점, p.154 "Set Zero") + `:set:cont 2 <z>`(현재 Z=contact 높이, p.139).
- `move_xy()` : `:mov:abs 2 X Y none` — **Z를 안전높이로 자동 분리한 뒤 XY 이동**(p.56)이라 이동 중 소자가 안 긁힌다.
- `contact()`/`separate()` : `:mov:cont`/`:mov:sep` — 사람이 set한 contact 높이로 복귀/분리. 그 아래로는 안 내려가 소자를 찍지 않는다.

In [4]:
class CascadeS300:
    def __init__(self, gpib_address=S300_GPIB):
        self.rm = pyvisa.ResourceManager()
        self.instrument = self.rm.open_resource(gpib_address)
        self.instrument.timeout = 30000      # 30 sec: 이동 완료(COMPLETE)까지 대기 (매뉴얼 권장)

    def _write(self, command):
        """메타 명령($:...) 등 '응답 없는' 명령 전용 — write 만."""
        self.instrument.write(command)

    def ask(self, command):
        """질의(? 명령) 또는 resp-on 상태의 명령. 응답 문자열 반환.
        $:set:resp on 상태에서 → 명령은 'COMPLETE', 질의는 값, 실패는 '@에러' 를 돌려줌."""
        try:
            return self.instrument.query(command).strip()
        except Exception as e:
            return f"Error: {e}"

    def cmd(self, command):
        """action/설정 명령 실행. $:set:resp on 덕에 완료되면 'COMPLETE' 반환(=완료까지 대기).
        응답이 '@' 로 시작하면 프로버 에러 → 예외 발생."""
        resp = self.ask(command)
        if resp.startswith("@"):
            raise RuntimeError(f"S300 명령 실패: {command!r} -> {resp!r}")
        return resp

    def setup(self):
        """매뉴얼 정식 원격 셋업 (Nucleus4 가이드 p.9). 순서 중요:
        $:set:resp on 을 먼저 켜야 이후 이동/설정 명령이 'COMPLETE' 응답을 줘서
        타임아웃 없이 완료를 확인할 수 있다. (이게 빠지면 모든 action 명령이 타임아웃)
        ※ 이 프로버 펌웨어는 :set: 명령에 device ID(CHUCK_ID)를 요구함(:set:unit 2 metric)."""
        self._write("$:set:mode summit")             # 명령 해석 모드 (meta, 응답없음)
        self._write("$:set:resp on")                 # ★ 명령마다 COMPLETE/에러 응답 켜기 (meta, 응답없음)
        self.cmd(":SYST:OPER:MODE REMOTE")           # 원격 모드 (device ID 불필요)
        self.cmd(f":set:unit {CHUCK_ID} metric")     # 단위 = micron (device ID 필요)
        return "OK (mode=summit, resp=on, REMOTE, metric)"

    # 하위호환 (개별 호출용)
    def set_remote(self):
        return self.cmd(":SYST:OPER:MODE REMOTE")

    def set_metric(self):
        return self.cmd(f":set:unit {CHUCK_ID} metric")   # device ID 필요

    # --- 위치 ---
    def read_position(self):
        """현재 척 좌표 (x, y, z) micron 반환 (:mov:abs? , p.62)."""
        resp = self.ask(f":mov:abs? {CHUCK_ID}")
        x, y, z = (float(v) for v in resp.replace(",", " ").split()[:3])
        return x, y, z

    def set_reference(self):
        """사람이 첫 소자에 contact 시킨 현재 위치를 원점(0,0)+contact높이로 등록.
        - 현재 XY → 원점 (:set:pres, p.154)   /   현재 Z → contact 높이 (:set:cont, p.139)"""
        x, y, z = self.read_position()
        self.cmd(f":set:pres {CHUCK_ID} 0 0")        # 현재 XY = 원점
        self.cmd(f":set:cont {CHUCK_ID} {int(z)}")   # 현재 Z = contact 높이
        print(f"기준점 등록: 현재위치 {(x, y, z)} → 원점(0,0), contact Z={int(z)}")

    # --- 분리 / 접촉 / 이동 (COMPLETE = 동작 완료. 별도 busy 폴링 불필요) ---
    def separate(self):
        """팁 분리 (:mov:sep, p.89). 완료(COMPLETE)까지 대기."""
        return self.cmd(f":mov:sep {CHUCK_ID}")

    def contact(self):
        """팁 접촉 — set된 contact 높이로 (:mov:cont, p.63). 완료까지 대기."""
        return self.cmd(f":mov:cont {CHUCK_ID}")

    def move_xy(self, dx, dy):
        """원점 기준 (dx,dy) micron 으로 XY 이동 (:mov:abs, z=none → Z 안전높이 자동분리).
        완료(COMPLETE)까지 대기 후 실제 위치 (x,y,z) 반환."""
        self.cmd(f":mov:abs {CHUCK_ID} {dx} {dy} none")
        return self.read_position()

## 2. B1500 측정 (② 코드)
`example_01.ipynb` 의 staircase sweep 을 함수로 묶고, **`SMU_CONFIG` 로 팁별 역할(sweep/const/off)을 선택**할 수 있게 함. `LINEAR_DOUBLE` = 왕복 sweep 이라 포인트는 `2*nop`.

In [5]:
def reset_gpib_controller():
    """GPIB Interface Clear(IFC) → PC 를 controller-in-charge 로. NCIC 예방/복구. 실패해도 무시."""
    try:
        _intf = pyvisa.ResourceManager().open_resource("GPIB0::INTFC")
        _intf.send_ifc()
        _intf.close()
        return True
    except Exception as e:
        print("  [IFC] 건너뜀:", type(e).__name__)
        return False


def drain_b1500_errors(b1500, max_reads=500):
    """B1500 FLEX 에러큐를 읽어서 끝까지 비운다(+0 'No Error' 나올 때까지).
    *CLS 로는 이 큐가 안 비워질 수 있어, 이전 통신오류(NCIC 등)로 쌓인
    +100 backlog 를 직접 제거한다. 비운 에러 개수를 반환."""
    n = 0
    for _ in range(max_reads):
        try:
            resp = b1500.ask("ERRX?")
        except Exception:
            return n                       # 통신오류(타임아웃 등) → 무한대기 방지, 즉시 종료
        if resp.split(",")[0].strip() in ("0", "+0"):
            return n
        n += 1
    return n


def setup_b1500():
    """B1500 연결 및 초기화"""
    # 참고: 최신 pymeasure(0.16+)의 AgilentB1500 는 read/write_termination 을
    # 내부에서 "\r\n" 으로 자동 설정하므로 여기서 넘기면 안 됨(중복 인자 에러).
    reset_gpib_controller()                       # ① NCIC 예방: GPIB 제어권 먼저 확보

    # ② 핸드셰이크는 짧은 타임아웃(8초) → 죽어 있으면 10분 대신 8초 안에 실패
    b1500 = AgilentB1500(B1500_GPIB, timeout=8000)
    b1500.clear()
    b1500.write("*CLS")
    idn = b1500.ask("*IDN?")
    if "B1500" not in idn:
        raise RuntimeError(
            f"B1500 응답 이상: {idn!r} "
            "→ 전원/GPIB 케이블 확인, NI MAX 에서 GPIB0 'System Controller' 체크 확인"
        )

    # ③ 응답 확인됐으니 측정용으로 타임아웃 늘림
    b1500.adapter.connection.timeout = 600000
    n = drain_b1500_errors(b1500)
    if n:
        print(f"[setup] B1500 에러큐에서 묵은 에러 {n}개 제거함")
    b1500.initialize_all_smus()
    b1500.data_format(21, mode=1)
    return b1500


def measure_iv(b1500, v_start, v_stop, nop, compliance, config):
    """config(SMU_CONFIG) 에 따라 각 SMU(=팁)를 sweep/const 로 설정하고 I-V 측정.
    반환: DataFrame (포인트 2*nop, LINEAR_DOUBLE 왕복 sweep)."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not sweep_chs:
        raise ValueError("sweep 할 SMU 가 없습니다. SMU_CONFIG 에서 하나는 drive='sweep' 이어야 함.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HRADC"
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.adc_setup("HRADC", "AUTO", 6)
    b1500.sweep_timing(0, 0.5, step_delay=0.1)        # hold, delay
    b1500.sweep_auto_abort(False, post="STOP")

    # 메인 sweep SMU (Gate 등)
    main_ch = sweep_chs[0]
    smus[main_ch].staircase_sweep_source(
        "VOLTAGE", "LINEAR_DOUBLE", "Auto Ranging",
        v_start, v_stop, nop, compliance,
    )
    # 추가로 sweep 하는 SMU 가 있으면 동기 sweep
    for ch in sweep_chs[1:]:
        smus[ch].synchronous_sweep_source("VOLTAGE", "Auto Ranging", v_start, v_stop, compliance)
    # const SMU (Drain/Source/Bulk 등) 는 DC 고정
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", config[ch]["const_v"], stepsize=0.1, pause=20e-3)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기
    b1500.check_idle()
    data = b1500.read_data(2 * nop)

    # const SMU 0V 로 복귀
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 2-C. `measure_transfer()` / `measure_output()`

둘 다 STAIRCASE_SWEEP 기반. 공통 저수준 `_measure_sweep()` 위에 만듦.
- `measure_transfer` → 게이트 sweep (Vd=VD_CONST 고정)
- `measure_output` → 드레인 sweep 을 VG_LIST 각 게이트값에서 반복 (family)

In [ ]:
def _measure_sweep(b1500, sweep_ch, v_start, v_stop, nop, const_volts, compliance,
                   meas_range="1 uA", direction="LINEAR_DOUBLE"):
    """sweep_ch 를 v_start~v_stop 로 sweep, const_volts={채널:전압} 는 DC 고정.
    STAIRCASE_SWEEP 로 측정, DataFrame 반환 (LINEAR_DOUBLE 이면 2*nop 점)."""
    const_chs = list(const_volts)
    active = [sweep_ch] + const_chs
    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HRADC"
        s.meas_range_current = meas_range
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.adc_setup("HRADC", "AUTO", 10)          # export: HRADC AUTO, coeff 10
    b1500.adc_auto_zero = False                   # export: AutoZero OFF
    b1500.sweep_timing(0, 0, step_delay=0)        # export: Hold=0, Delay=0
    b1500.sweep_auto_abort(False, post="START")   # export: CONTINUE AT ANY, PostOutput=START

    smus[sweep_ch].staircase_sweep_source(
        "VOLTAGE", direction, "Auto Ranging", v_start, v_stop, nop, compliance)
    for ch, v in const_volts.items():
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", v, stepsize=0.1, pause=20e-3)

    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()
    b1500.check_idle()
    npts = 2*nop if "DOUBLE" in direction else nop
    data = b1500.read_data(npts)

    for ch in const_volts:            # 고정 소스 0V 복귀
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data


def measure_transfer(b1500):
    """Transfer curve (Id-Vg): 게이트 sweep 을 VD_LIST 각 드레인값에서 반복 (family).
    반환: 세로로 이어붙인 DataFrame (맨 앞에 그때의 Vd 컬럼)."""
    frames = []
    for vd in VD_LIST:
        d = _measure_sweep(
            b1500, GATE_CH, VG_START, VG_STOP, VG_POINTS,
            {DRAIN_CH: vd, SOURCE_CH: 0.0}, TR_COMPLIANCE,
            meas_range=TR_MEAS_RANGE)
        d.insert(0, "Vd step (V)", vd)
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


def measure_output(b1500):
    """Output curve (Id-Vd): 드레인 sweep 을 VG_LIST 각 게이트값에서 반복.
    반환: 세로로 이어붙인 DataFrame (맨 앞에 그때의 Vg 컬럼)."""
    frames = []
    for vg in VG_LIST:
        d = _measure_sweep(
            b1500, DRAIN_CH, VD_START, VD_STOP, VD_POINTS,
            {GATE_CH: vg, SOURCE_CH: 0.0}, OUT_COMPLIANCE,
            meas_range=OUT_MEAS_RANGE)
        d.insert(0, "Vg step (V)", vg)
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

## 2-B. `measure_iv_full()` — Measurement Setup 을 그대로 반영한 I-V 측정

원본 `measure_iv` 는 direction/ADC/range/timing 이 **하드코딩**돼 있다. 
`measure_iv_full` 은 위 `VAR1_CONFIG / VAR2_CONFIG / TIMING_CONFIG / ADC_CONFIG / RANGE_CONFIG` 를 읽어
**UI 손잡이를 코드로 그대로 설정**한다. 역할/Constants 는 원본 `SMU_CONFIG` 를 재사용.

- **VAR1** → `staircase_sweep_source` (direction+linear/log → SweepMode, start/stop/points/compliance/Pwr Comp)
- **VAR2** → 파이썬 외부 루프 (각 스텝마다 VAR2 채널 DC 를 바꿔 VAR1 sweep 반복 → 세로로 이어붙임)
- **Timing** → `sweep_timing` + `sweep_auto_abort`
- **Ranging** → 채널별 `meas_range_current` (auto/limited/fixed)

> 원본 `measure_iv`/`measure_sampling` 은 그대로 남아있어 기존 셀들도 문제없이 동작한다.

In [6]:
def _sweep_mode(direction, spacing):
    """UI Direction(single/double) + Linear/Log -> pymeasure SweepMode 이름."""
    table = {
        ("single", "linear"): "LINEAR_SINGLE",
        ("double", "linear"): "LINEAR_DOUBLE",
        ("single", "log"):    "LOG_SINGLE",
        ("double", "log"):    "LOG_DOUBLE",
    }
    key = (direction.lower(), spacing.lower())
    if key not in table:
        raise ValueError(f"direction/spacing 조합 오류: {direction}/{spacing}")
    return table[key]


def _range_name(mode, rng):
    """UI Ranging Mode(auto/limited/fixed)+레인지값 -> meas_range_current 이름."""
    m = mode.lower()
    if m == "auto":
        return "Auto Ranging"
    if m == "limited":
        return f"{rng} limited auto ranging"
    if m == "fixed":
        return f"{rng} range fixed"
    raise ValueError(f"ranging mode 오류: {mode!r} (auto|limited|fixed 중 하나)")


def _var_points(cfg):
    """points 우선. step 이 지정되면 No of Step = round(|stop-start|/step)+1 로 환산."""
    if cfg.get("step"):
        return int(round(abs(cfg["stop"] - cfg["start"]) / cfg["step"])) + 1
    return int(cfg["points"])


def _var_values(cfg):
    """VAR2 스텝 전압 리스트 (선형)."""
    n = _var_points(cfg)
    if n <= 1:
        return [cfg["start"]]
    return [cfg["start"] + (cfg["stop"] - cfg["start"]) * k / (n - 1) for k in range(n)]


def measure_iv_full(b1500, config=None, var1=None, var2="__default__",
                    timing=None, adc=None, ranges=None):
    """EasyEXPERT 'Measurement Setup'(VAR1/VAR2/Timing/Constants/Ranging)을 코드로 재현한 I-V 측정.

    인자는 None(또는 var2 는 "__default__") 이면 위 CONFIG 전역값을 사용. 개별 오버라이드 가능.
    - config : SMU_CONFIG (drive=sweep/const/off, const_v) — 역할 + Constants 정의
    - var1   : VAR1_CONFIG — 주 sweep 상세
    - var2   : VAR2_CONFIG or None — 부 sweep. None 이면 VAR2 미사용(=VAR1 1회)
    - timing/adc/ranges : TIMING/ADC/RANGE_CONFIG
    반환: DataFrame. VAR2 사용 시 맨 앞에 VAR2 값 컬럼이 붙고 스텝별 결과가 세로로 이어붙음.
    """
    config = SMU_CONFIG   if config is None else config
    var1   = VAR1_CONFIG  if var1   is None else var1
    var2   = VAR2_CONFIG  if var2 == "__default__" else var2
    timing = TIMING_CONFIG if timing is None else timing
    adc    = ADC_CONFIG   if adc    is None else adc
    ranges = RANGE_CONFIG if ranges is None else ranges

    sweep_ch = var1["unit"]
    if config.get(sweep_ch, {}).get("drive") != "sweep":
        raise ValueError(f"VAR1 unit(SMU{sweep_ch}) 은 SMU_CONFIG 에서 drive='sweep' 이어야 함.")

    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = [sweep_ch] + const_chs
    smus      = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    mode = _sweep_mode(var1["direction"], var1["spacing"])
    nop  = _var_points(var1)
    npts = nop if "SINGLE" in mode else 2 * nop      # 왕복(DOUBLE)이면 포인트 2배

    # VAR2 준비 (없으면 [None] 로 1회만)
    if var2:
        var2_ch = var2["unit"]
        if var2_ch not in const_chs:
            raise ValueError(f"VAR2 unit(SMU{var2_ch}) 은 SMU_CONFIG 에서 drive='const' 여야 함.")
        v2_vals = _var_values(var2)
    else:
        var2_ch, v2_vals = None, [None]

    frames = []
    for v2 in v2_vals:
        # --- 측정 모드/채널 (MM) ---
        b1500.meas_mode("STAIRCASE_SWEEP", *[smus[ch] for ch in active])
        for ch in active:
            s = smus[ch]
            s.enable()
            s.adc_type     = adc["adc_type"]
            s.meas_op_mode = "COMPLIANCE_SIDE"

        # --- Ranging Mode (RI) : RANGE_CONFIG 있는 채널만 ---
        for ch in active:
            rc = (ranges or {}).get(ch)
            if rc:
                smus[ch].meas_range_current = _range_name(rc["mode"], rc.get("range"))

        # --- ADC / Timing ---
        b1500.adc_setup(adc["adc_type"], adc["mode"], adc["N"])
        b1500.sweep_timing(timing["hold"], timing["delay"], step_delay=timing["step_delay"])
        b1500.sweep_auto_abort(timing["auto_abort"], post=timing["post"])

        # --- VAR1 sweep 소스 (WV) ---
        pcomp = "" if var1.get("power_comp") in (None, "") else var1["power_comp"]
        smus[sweep_ch].staircase_sweep_source(
            "VOLTAGE", mode, "Auto Ranging",
            var1["start"], var1["stop"], nop, var1["compliance"], pcomp,
        )
        # --- Constants (DV) : VAR2 채널이면 이번 스텝 값으로 덮어씀 ---
        for ch in const_chs:
            val = v2 if (ch == var2_ch) else config[ch]["const_v"]
            smus[ch].ramp_source("VOLTAGE", "Auto Ranging", val, stepsize=0.1, pause=20e-3)

        # --- 측정 실행 ---
        b1500.check_errors()
        b1500.clear_buffer()
        b1500.clear_timer()
        b1500.send_trigger()
        b1500.check_idle()
        data = b1500.read_data(npts)

        if var2_ch is not None:
            data.insert(0, f"VAR2 SMU{var2_ch} {var2.get('name','V')} (V)", v2)
        frames.append(data)

    # 모든 const SMU 0V 복귀
    for ch in const_chs:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)

    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]


## 2.5 (선택) Sampling 측정 — 시간에 따른 전류 (transient / 안정성)

`measure_iv` 는 전압을 **쓸면서**(sweep) I-V 곡선을 얻고, `measure_sampling` 은 전압을 **고정**한 채 일정 시간 간격마다 전류를 찍어 **시간축(I-t) 데이터**를 얻습니다. (B1500 매뉴얼의 `SAMPLING` 모드)

- 같은 `SMU_CONFIG` 를 그대로 재사용 — `sweep` 역할 SMU(GATE)는 `V_STOP`(또는 `SAMP_SWEEP_BIAS`)로 고정, `const` 역할은 `const_v` 로 고정, `off` 는 미사용.
- 반환 `DataFrame` 에는 **time stamp 열 + 채널별 전류 열**이 들어 있어 `전류 vs 시간` 으로 바로 그릴 수 있음.
- 용도: bias-stress 안정성, 접촉 안정화 확인, drift 모니터링 등. **필요 없으면 호출 안 하면 됨** (I-V 측정엔 영향 없음).

> ⚠️ `measure_iv`(STAIRCASE_SWEEP) 와 `measure_sampling`(SAMPLING) 은 측정 모드를 서로 바꾸므로, 한 좌표에서 **둘 다** 쓸 경우 순서대로(예: I-V 먼저 → sampling 나중) 호출하면 됩니다. 각자 자기 모드를 다시 설정하므로 충돌 없음.

In [ ]:
# --- Sampling(시간) 측정 파라미터 (필요할 때만) -------------------------------
# I-V(sweep)와 달리 전압을 '고정'해두고 일정 시간 간격마다 전류를 찍어
# 시간에 따른 변화(transient / bias-stress / 안정성)를 본다.
SAMP_INTERVAL  = 0.01    # 샘플 간격 [s]  (>=0.002 권장, 그 미만은 고속/제약 있음)
SAMP_NUMBER    = 200     # 샘플 개수  (총 측정시간 ≈ INTERVAL * NUMBER)
SAMP_HOLD_BIAS = 0       # bias 인가 후 첫 측정까지 대기시간 [s]
SAMP_BASE_V    = 0.0     # base(측정 전/후) 전압 [V]

# sweep 역할 SMU(예: GATE)를 sampling 중엔 어떤 DC 전압으로 고정할지.
# None 이면 V_STOP('on' 전압)을 사용. const 역할은 SMU_CONFIG 의 const_v 그대로.
SAMP_SWEEP_BIAS = None   # 예: 1.0 으로 두면 게이트를 1V 고정한 채 시간 측정


def measure_sampling(b1500, config, interval=SAMP_INTERVAL, number=SAMP_NUMBER,
                     hold_bias=SAMP_HOLD_BIAS, base=SAMP_BASE_V,
                     sweep_bias=SAMP_SWEEP_BIAS, compliance=I_COMPLIANCE):
    """SAMPLING 모드: 각 SMU(=팁)에 DC bias 를 걸고 '시간에 따른 전류'를 측정.
    - sweep 역할 SMU(GATE 등) → sweep_bias(None 이면 V_STOP)로 고정
    - const 역할 SMU         → SMU_CONFIG 의 const_v 로 고정
    - off                    → 사용 안 함
    반환: DataFrame (number 포인트, time stamp + 채널별 전류).
    measure_iv 와 같은 SMU_CONFIG 를 그대로 재사용한다."""
    sweep_chs = [ch for ch, c in config.items() if c["drive"] == "sweep"]
    const_chs = [ch for ch, c in config.items() if c["drive"] == "const"]
    active    = sweep_chs + const_chs
    if not active:
        raise ValueError("sampling 할 SMU 가 없습니다. SMU_CONFIG 확인.")

    smus = {ch: getattr(b1500, f"smu{ch}") for ch in active}

    # 측정 모드 = SAMPLING (측정 순서 = active 순서)
    b1500.meas_mode("SAMPLING", *[smus[ch] for ch in active])
    for ch in active:
        s = smus[ch]
        s.enable()
        s.adc_type = "HSADC"            # sampling 은 고속 ADC 사용
        s.meas_range_current = "1 uA"
        s.meas_op_mode = "COMPLIANCE_SIDE"

    b1500.sampling_mode = "LINEAR"
    b1500.adc_setup("HSADC", "AUTO", 1)
    b1500.sampling_timing(hold_bias, interval, number)   # MT: hold, interval, #points
    b1500.sampling_auto_abort(False, post="Bias")        # 중도중단 off
    b1500.time_stamp = True                              # 시간축 기록 ON

    # 각 SMU 에 DC bias 인가 (base → bias)
    for ch in active:
        bias = (sweep_bias if sweep_bias is not None else V_STOP) if ch in sweep_chs \
            else config[ch]["const_v"]
        smus[ch].sampling_source("VOLTAGE", "Auto Ranging", base, bias, compliance)

    # 측정 시작
    b1500.check_errors()
    b1500.clear_buffer()
    b1500.clear_timer()
    b1500.send_trigger()

    # 끝날 때까지 대기 후 한 번에 읽기 (time stamp + current 포함)
    b1500.check_idle()
    data = b1500.read_data(number)

    # 모든 SMU 0V 로 복귀
    for ch in active:
        smus[ch].ramp_source("VOLTAGE", "Auto Ranging", 0, stepsize=0.1, pause=20e-3)
    return data

## 3. 장비 연결

오류가 자주 나는데, vscode를 껐다 키고, B1500을 껐다 키고, 잭 연결이 잘 되어있는지 확인해보기

In [ ]:
s300 = CascadeS300(S300_GPIB)
s300.instrument.timeout = 5000            # 핸드셰이크는 짧게(빠른 실패)
print("S300:", s300.ask("*IDN?"))
print("S300 setup:", s300.setup())        # mode summit + resp on + REMOTE + metric
s300.instrument.timeout = 30000           # 이동용으로 원복 (큰 이동 대비)

b1500 = setup_b1500()                      # 내부에서 IFC + 짧은 타임아웃 핸드셰이크 → 빠른 실패
print("B1500:", b1500.ask("*IDN?"))

S300: Cascade Microtech, S300 Theta, 586480504, 3, 3
S300 setup: OK (mode=summit, resp=on, REMOTE, metric)


## 3.5 통신 확인 (S300 ↔ B1500)
좌표 이동(S300)·측정(B1500) 두 장비가 모두 응답하는지, S300 이 REMOTE/정렬 완료 상태인지 먼저 확인합니다.
**[OK] 가 떠야 다음으로 진행하세요.** (매뉴얼 p.9 `Verifying GPIB Communication` 절에 해당)

In [35]:
def check_comm(s300, b1500):
    """S300(좌표이동) + B1500(측정) 통신/준비상태 확인."""
    print("=== 통신 확인 ===")
    ok = True

    # --- S300 ---
    idn = s300.ask("*IDN?")
    print("S300 *IDN?        :", idn)
    ok &= "Cascade" in idn

    mode = s300.ask("$:set:mode?")          # SUMMIT/EG = Nucleus 인터프리터 동작중
    print("S300 interpreter  :", mode)
    ok &= mode in ("SUMMIT", "EG")

    tst = s300.ask("*tst?")                 # 0 = self-test 정상
    print("S300 self-test    :", tst)
    ok &= tst.strip() == "0"

    s300.set_remote()
    oper = s300.ask(":SYST:OPER:MODE?")     # REMOTE 여야 원격제어 가능
    print("S300 oper mode    :", oper)
    ok &= oper == "REMOTE"

    align = s300.ask(":align:wafer:busy?")  # SUCCESS = 수동 alignment 끝난 상태인지 확인용
    print("S300 align status :", align)     # (정렬 안 됐으면 좌표 이동이 어긋남)

    # --- B1500 ---
    bidn = b1500.ask("*IDN?")
    print("B1500 *IDN?       :", bidn)
    ok &= "B1500" in bidn

    opc = b1500.ask("*OPC?")                # 1 = idle/완료
    print("B1500 *OPC?       :", opc)
    ok &= opc.strip() == "1"

    print("B1500 UNT?        :", b1500.ask("UNT?"))   # 장착 모듈 목록

    print("\n결과:", "[OK] 통신 정상" if ok else "[FAIL] 확인 필요 - 위 항목 점검")
    return ok


check_comm(s300, b1500)

=== 통신 확인 ===
S300 *IDN?        : Cascade Microtech, S300 Theta, 586480504, 3, 3
S300 interpreter  : SUMMIT
S300 self-test    : 0
S300 oper mode    : REMOTE
S300 align status : SUCCESS
B1500 *IDN?       : Agilent Technologies,B1500A,0,A.06.02.2023.0401
B1500 *OPC?       : 1
B1500 UNT?        : B1530A,0;B1517A,0;B1517A,0;B1517A,0;0,0;0,0;0,0;0,0;0,0;0,0

결과: [OK] 통신 정상


True

## 4. 좌표 파일 읽기 (CSV / 엑셀 자동 판별)
`COORD_CSV` 의 확장자가 `.csv` 면 `read_csv`, `.xlsx/.xls` 면 `read_excel` 로 읽습니다. 둘 다 `Subsite Name, X Position, Y Position` 컬럼 구조는 동일.

In [36]:
# 확장자 보고 CSV / 엑셀 자동 판별 (.xlsx 는 openpyxl 필요: pip install openpyxl)
ext = os.path.splitext(COORD_CSV)[1].lower()
if ext in (".xlsx", ".xls"):
    coords = pd.read_excel(COORD_CSV)
else:
    coords = pd.read_csv(COORD_CSV)

print(f"좌표 {len(coords)} 개 로드 완료: {COORD_CSV}")
coords.head()

좌표 3 개 로드 완료: ../../utils/test_coordinates.csv


,Subsite Name,X Position,Y Position,Note
0,1,0,0,origin (사람이 contact시킨 자리 - 이동없음)
1,2,100,0,오른쪽 100um
2,3,0,100,위 100um


## 5. 기준점 등록 — ⚠️ 사람이 첫 소자에 팁 contact 시킨 상태에서 실행
위 '준비' 단계대로 **사람이 첫 소자에 팁을 직접 contact** 시킨 상태에서 아래 셀을 실행하세요.
현재 위치가 모든 좌표의 원점(0,0)·contact 높이로 등록됩니다 (실제 contact 위치 기준이라 이후 소자를 찍지 않음).

In [37]:
s300.set_reference()

기준점 등록: 현재위치 (0.0, 0.0, 3000.0) → 원점(0,0), contact Z=3000


## 6. 메인 루프 — 좌표대로 쭉 이동 + 측정 (① + ② 합치는 곳)

**좌표 convention**: 파일의 X/Y 는 **원점(사람이 contact시킨 소자=0,0) 기준 상대좌표**이고 음수도 가능 (예: 좌상단 `-100, 100`). 보통은 처음 찍는 소자를 `0,0` 으로 둔다.

- **좌표 (0,0)** 인 줄 = 원점 = 사람이 이미 contact → 이동 없이 바로 측정
- **그 외** = `separate()`(분리) → `move_xy()`(XY 이동, Z 자동 분리) → `contact()`(접촉) → 측정

> 💡 처음엔 측정 없이 **이동만** 확인하려면 아래 `measure_iv(...)` 와 저장 줄을 주석 처리하세요.

In [41]:
os.makedirs(OUT_DIR, exist_ok=True)

# --- 팁 보호: Z 축만 느리게 (XY 속도는 건드리지 않음 = 장비 기본값 유지) ---
s300.cmd(":set:cont:spee 100")              # 접촉(올림) 100 µm/s (기본 500 → 느리게)
s300.cmd(f":set:vel {CHUCK_ID} z slow")     # z(내림/복귀)만 느리게. xy 는 손대지 않음
print("팁보호 속도: 접촉", s300.ask(":set:cont:spee?"), "µm/s, z=slow (xy 기본값 유지)")

# 측정 종류 선택: "transfer"=Id-Vg | "output"=Id-Vd | "both"=둘 다
MEAS_TYPE = "both"       # transfer(Id-Vg) + output(Id-Vd) 둘 다

assert MEAS_TYPE in ("transfer", "output", "both"), 'MEAS_TYPE 는 transfer/output/both 중 하나'
do_transfer = MEAS_TYPE in ("transfer", "both")
do_output   = MEAS_TYPE in ("output", "both")
print(f"측정 종류: {MEAS_TYPE}  (transfer={do_transfer}, output={do_output})")

# ⚠️ 이동거리 배율 (테스트용). test_coordinates.csv 는 100µm라 눈에 안 보여서 확대해 확인.
#    실제 좌표 파일(grid_coordinates.csv)로 측정할 땐 반드시 1 로 되돌릴 것!
COORD_SCALE = 1     # 실제 측정: CSV 좌표 그대로 사용 (테스트 땐 100 이었음)
print(f"이동거리 배율: ×{COORD_SCALE}")
LIFT_Z = 2000   # 지점 간 이동 시 척을 "아래로" 내릴 분리량 [µm] (2mm)


for _, row in coords.iterrows():
    sub_id = row["Subsite Name"]
    dx = int(row["X Position"]) * COORD_SCALE   # 원점 기준 상대좌표 × 배율 (음수 가능)
    dy = int(row["Y Position"]) * COORD_SCALE

    if dx == 0 and dy == 0:
        # 좌표 (0,0) = 원점 = 사람이 이미 contact 시킨 소자 → 이동 없이 바로 측정
        print(f"[Subsite {sub_id}] 원점 (0, 0) — 이동 없음")
    else:
        print(f"[Subsite {sub_id}] -> 원점+({dx}, {dy}) 이동")
        # 확정 방향: z↑=척 위(팁 쪽) / z↓=척 아래(팁에서 멀어짐)
        # 순서: ① 분리(척 아래로) → ② XY 이동(분리된 채) → ③ 접촉(척 위로)
        xc, yc, zc = s300.read_position()
        s300.cmd(f":mov:abs {CHUCK_ID} {int(xc)} {int(yc)} {int(zc) - LIFT_Z}")  # ① 분리: 아래로(z↓)
        print("    ① 분리(척 아래로):", s300.read_position())
        s300.move_xy(dx, dy)                                                     # ② XY 이동 (분리된 채)
        print("    ② 이동:", s300.read_position())
        s300.contact()                                                          # ③ 접촉 (척 위로 올려 팁 닿음)
        print("    ③ 접촉(척 위로):", s300.read_position())
        time.sleep(0.2)



    # ① Transfer curve (Id-Vg) — 게이트 sweep
    if do_transfer:
        data = measure_transfer(b1500)
        p = os.path.join(OUT_DIR, f"subsite_{sub_id}_transfer.csv")
        data.to_csv(p)
        print(f"  저장(transfer): {p}")

    # ② Output curve (Id-Vd) — 드레인 sweep (VG_LIST family)
    if do_output:
        data = measure_output(b1500)
        p = os.path.join(OUT_DIR, f"subsite_{sub_id}_output.csv")
        data.to_csv(p)
        print(f"  저장(output): {p}")

팁보호 속도: 접촉 100 µm/s, z=slow (xy 기본값 유지)
측정 모드: iv  (I-V=True, sampling=False)
이동거리 배율: ×100
[Subsite 1] 원점 (0, 0) — 이동 없음
  저장(I-V): results\subsite_1.csv
[Subsite 2] -> 원점+(10000, 0) 이동
    ① 분리: (0.0, 10000.0, 5000.0)
    ② 이동: (10000.0, 0.0, 5000.0)
    ③ 접촉: (10000.0, 0.0, 3000.0)
  저장(I-V): results\subsite_2.csv
[Subsite 3] -> 원점+(0, 10000) 이동
    ① 분리: (10000.0, 0.0, 5000.0)
    ② 이동: (0.0, 10000.0, 5000.0)
    ③ 접촉: (0.0, 10000.0, 3000.0)
  저장(I-V): results\subsite_3.csv


## 6-B. (대안) 메인 루프 — `measure_iv_full()` 사용

위 `## 6` 원본 루프는 `measure_iv`(하드코딩)를 쓴다. 아래 루프는 **똑같은 좌표이동**에
측정만 `measure_iv_full`(설정판 반영)로 바꾼 것. 둘 중 하나만 실행하면 된다.

- `VAR1_CONFIG` 로 sweep 조건 결정. `VAR2_CONFIG` 지정 시 좌표당 2차원 sweep.
- 결과는 `results_config/` 에 저장 (원본 `results/` 와 분리).

In [ ]:
OUT_DIR_CFG = "results_config"   # 설정판 결과 폴더 (원본 results/ 와 분리)
os.makedirs(OUT_DIR_CFG, exist_ok=True)

print("VAR1:", VAR1_CONFIG["name"],
      f'{VAR1_CONFIG["start"]}~{VAR1_CONFIG["stop"]}V',
      f'{_var_points(VAR1_CONFIG)}pts', VAR1_CONFIG["direction"], VAR1_CONFIG["spacing"])
print("VAR2:", "미사용" if not VAR2_CONFIG else
      f'{VAR2_CONFIG["name"]} {VAR2_CONFIG["start"]}~{VAR2_CONFIG["stop"]}V {_var_points(VAR2_CONFIG)}스텝')

for _, row in coords.iterrows():
    sub_id = row["Subsite Name"]
    dx = int(row["X Position"])
    dy = int(row["Y Position"])

    if dx == 0 and dy == 0:
        print(f"[Subsite {sub_id}] 원점 (0, 0) — 이동 없음")
    else:
        print(f"[Subsite {sub_id}] -> 원점+({dx}, {dy}) 이동")
        s300.separate()
        s300.move_xy(dx, dy)
        s300.contact()
        time.sleep(0.2)

    # 설정판(VAR1/VAR2/Timing/Ranging) 을 그대로 반영한 I-V 측정
    data = measure_iv_full(b1500)
    out_path = os.path.join(OUT_DIR_CFG, f"subsite_{sub_id}.csv")
    data.to_csv(out_path)
    print(f"  저장: {out_path}  ({len(data)} 행)")


## 7. 마무리 — 팁 분리 후 원점 복귀

In [ ]:
s300.separate()      # 팁 분리
s300.move_xy(0, 0)   # 원점으로 복귀 (Z 자동 분리 상태로 이동)
print("측정 완료.")

In [9]:
# --- 종료: S300 LOCAL 복귀 + 세션 정리 -------------------------------------
# 케이블 뽑기/세션 끝내기 전에 실행. (척을 움직이지 않는 안전한 명령)
print("S300 -> LOCAL:", s300.ask(":SYST:OPER:MODE LOCAL"))   # 원격모드 해제 (UI 수동조작 복귀)

# VISA 세션 닫기 (생략해도 커널 Restart 시 자동 해제됨)
try:
    s300.instrument.close()
    b1500.adapter.close()
    print("VISA 세션 닫음.")
except Exception as e:
    print("close 중 예외(무시 가능):", e)


S300 -> LOCAL: Error: VI_ERROR_NCIC (-1073807264): The interface associated with this session is not currently the controller in charge.
close 중 예외(무시 가능): name 'b1500' is not defined


In [ ]:
import time
D = 10000   # 이동 거리 [µm] = 1cm  (더 키우려면 이 숫자만 바꾸기)
print("시작:", s300.read_position())
for target in [(D, 0), (D, D), (0, D), (0, 0)]:
    print(f"→ 이동 {target}:", s300.move_xy(*target))
    time.sleep(2)   # 각 모서리 2초 멈춤 — 눈으로 확인


In [12]:
import time, os
os.makedirs(OUT_DIR, exist_ok=True)

D = 10000   # 이동 거리 [µm] = 1cm  (숫자만 바꾸면 조절)
targets = [(0, 0), (D, 0), (D, D), (0, D), (0, 0)]   # 원점→오른쪽→위→왼쪽→원점

print("시작 위치:", s300.read_position())
for i, (dx, dy) in enumerate(targets):
    pos = s300.move_xy(dx, dy)                 # ① XY 이동 (팁 비접촉, Z는 안전높이)
    print(f"[{i}] 이동 {(dx, dy)} → 실제위치 {pos}")

    data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)  # ② 측정 (개방→≈0)
    path = os.path.join(OUT_DIR, f"movemeas_{i}_{dx}_{dy}.csv")
    data.to_csv(path)
    print(f"     측정 저장: {path}  (전류≈0 예상, {len(data)}포인트)")

    time.sleep(1)   # 눈으로 확인용 잠깐 멈춤
print("완료")


시작 위치: (0.0, 0.0, 1000.0)
[0] 이동 (0, 0) → 실제위치 (0.0, 0.0, 1000.0)
     측정 저장: results\movemeas_0_0_0.csv  (전류≈0 예상, 22포인트)
[1] 이동 (10000, 0) → 실제위치 (10000.0, -1.0, 1000.0)
     측정 저장: results\movemeas_1_10000_0.csv  (전류≈0 예상, 22포인트)
[2] 이동 (10000, 10000) → 실제위치 (10000.0, 10000.0, 1000.0)
     측정 저장: results\movemeas_2_10000_10000.csv  (전류≈0 예상, 22포인트)
[3] 이동 (0, 10000) → 실제위치 (0.0, 10000.0, 1000.0)
     측정 저장: results\movemeas_3_0_10000.csv  (전류≈0 예상, 22포인트)
[4] 이동 (0, 0) → 실제위치 (0.0, 2.0, 1000.0)
     측정 저장: results\movemeas_4_0_0.csv  (전류≈0 예상, 22포인트)
완료


In [ ]:
# ── Z축 이동 테스트 : 팁에서 300µm 멀어졌다 복귀 ──────────────────────
#  이 프로버는 z 가 거꾸로: z 증가 = 척 아래로(팁에서 멀어짐) = ✅안전
#                          z 감소 = 팁 쪽 = ⚠️위험 (쓰지 말 것)

def move_z_abs(z):
    """현재 X,Y 유지한 채 Z만 절대이동 (:mov:abs 2 x y z). 완료까지 대기."""
    x, y, _ = s300.read_position()
    s300.cmd(f":mov:abs {CHUCK_ID} {int(x)} {int(y)} {int(z)}")
    return s300.read_position()

DZ = 300   # 팁에서 멀어질 양 [µm] = 0.3mm (멀어지는 방향이라 커도 안전)

x0, y0, z0 = s300.read_position()
print("시작            :", (x0, y0, z0))
print(f"→ 팁에서 {DZ}um 멀어짐:", move_z_abs(z0 + DZ))   # +z = 멀어짐(안전)
time.sleep(1)                                            # 눈으로 확인
print("→ 원위치 복귀    :", move_z_abs(z0))               # 시작 높이로
print("Z 테스트 완료.")




시작(접촉)     : (0.0, 0.0, 1.0)
→ 팁에서 30um 멀어짐: (0.0, 0.0, 31.0)
→ 접촉으로 복귀 : (0.0, 0.0, 1.0)
Z 테스트 완료.


In [ ]:
import time

# 확인됨(실측): z 증가 = 척 위로 / z 감소 = 척 아래로
DOWN = -1   # 아래로 = z 감소

x, y, z = (int(v) for v in s300.read_position())
print(f"시작 z = {z} — 척을 아래로 최대한 내립니다")

for step in (2000, 500, 100):          # 큰 폭 → 점점 작게 (리밋에 최대한 근접)
    while True:
        try:
            s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} {z + DOWN*step}")
        except RuntimeError:
            break                       # 이 폭으론 더 못 감 → 다음(더 작은) 폭으로
        z = int(s300.read_position()[2])
        print(f"   (step {step}) z = {z}")
        time.sleep(0.2)

print("최하단 도달. z =", s300.read_position()[2])


ValueError: could not convert string to float: 'Error:'

In [54]:
x, y, z = s300.read_position()
print(f"현재 위치: X={x}, Y={y}, Z={z}")
print(f"현재 Z = {z} µm")


ValueError: could not convert string to float: 'Error:'

In [50]:
x, y, z = (int(v) for v in s300.read_position())
print("시작 z =", z)

s300.cmd(f":mov:abs {CHUCK_ID} {x} {y} 5000")   # Z를 5000 으로 (XY 그대로)

print("이동후 z =", s300.read_position()[2])


시작 z = 10201


ValueError: could not convert string to float: 'COMPLETE'

In [ ]:
# ── 이동 없이 "지금 접촉된 자리"만 측정 + 저장 ─────────────────────────
#  사람이 팁을 접촉시켜 둔 상태 그대로. XY/Z 이동 명령 전혀 없음.

import os
from datetime import datetime

SAMPLE_NAME = "sample1"           # 파일명에 들어갈 이름 (원하는 대로 변경)
OUT_DIR_ONE = "results_single"    # 저장 폴더
os.makedirs(OUT_DIR_ONE, exist_ok=True)

print("현재 위치(이동 안 함):", s300.read_position())   # 확인용 조회만 (움직이지 않음)

# 측정 — 설정 셀의 V_START/V_STOP/V_POINTS/I_COMPLIANCE/SMU_CONFIG 그대로 사용
data = measure_iv(b1500, V_START, V_STOP, V_POINTS, I_COMPLIANCE, SMU_CONFIG)

# 저장 (실행할 때마다 시각이 붙어 덮어쓰지 않음)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path = os.path.join(OUT_DIR_ONE, f"{SAMPLE_NAME}_{stamp}.csv")
data.to_csv(path)

print(f"저장 완료: {path}   ({len(data)} 행)")
print("컬럼:", list(data.columns))
data.head()
